# SLSQP vs L-BFGS (windowed vs full-grid, CPU vs GPU)

End-to-end head-to-head for every corrector in `dvfopt`:

| Method | 2D entry | 3D entry |
|---|---|---|
| SLSQP windowed | `iterative_serial` | `iterative_3d` |
| SLSQP full-grid | inline `scipy.optimize.minimize(method='SLSQP', ...)` | **skipped** (infeasible in 3D — 3N² constraint-Jacobian cost) |
| L-BFGS windowed (CPU) | `iterative_2d_barrier` | `iterative_3d_barrier` |
| L-BFGS full-grid (CPU) | `iterative_2d_barrier(windowed=False)` | `iterative_3d_barrier(windowed=False)` |
| L-BFGS windowed (GPU) | `iterative_2d_barrier_torch` | `iterative_3d_barrier_torch` |
| L-BFGS full-grid (GPU) | `iterative_2d_barrier_torch(windowed=False)` | `iterative_3d_barrier_torch(windowed=False)` |

Metrics per (case, method): wall time, final neg-Jdet count, final min Jdet, L2 distortion `||phi - phi_init||_2`.

**Test data** (used in tandem, not instead of each other):

- **Synthetic random DVFs** — same `generate_random_dvf` / `generate_random_dvf_3d` baseline the rest of the repo uses. Labeled `rand_*`.
- **Structured 2D cases** from `test_cases` simulating localized folds — crossings (`01a`, `03b`), opposite-motion (`01b`, `03c`), edge discontinuities (`01c`).
- **Real registration output** — 2D slices (`02a_64x91`, `02c_64x91`, `02a_320x456`) and the 3D volume (`real_8` … `real_2`, factors 1/8 to 1/2 of shape `(528, 320, 456)`). Real folds cluster spatially near structure boundaries and look nothing like i.i.d. noise.

**Skip policy.** Full-grid SLSQP capped at ≤30×30 (2D) and skipped entirely for 3D (3·N² dense constraint Jacobian). Windowed SLSQP runs on every case in both dims; 2D is reliable (bounding-window QPs stay small), 3D can hang above real_8 because one extra dim plus `+2·pad` pushes the first QP into active-set cycling territory. L-BFGS variants run everywhere, subject only to GPU VRAM caps.

In [1]:
import os, sys, time
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.optimize import minimize, NonlinearConstraint

from test_cases import make_deformation
from test_cases._builders import load_slice
from dvfopt import (
    iterative_serial, iterative_3d,
    jacobian_det2D, jacobian_det3D,
    generate_random_dvf, generate_random_dvf_3d,
    scale_dvf, scale_dvf_3d,
    DEFAULT_PARAMS,
)
from dvfopt.core.iterative2d_barrier import iterative_2d_barrier, iterative_2d_barrier_torch
from dvfopt.core.iterative3d_barrier import iterative_3d_barrier
from dvfopt.core.iterative3d_barrier_torch import iterative_3d_barrier_torch
from dvfopt.jacobian.numpy_jdet import _numpy_jdet_2d, _numpy_jdet_3d
from dvfopt.core.objective import objective_euc

from benchmark_utils import (
    get_output_dir, save_results_csv, save_summary_json,
    log_run_header, log_run_footer, show_and_save, reset_figure_counter,
)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

CUDA: True
Device: NVIDIA GeForce RTX 3050 OEM


In [2]:
METHOD = 'comparison'
NOTEBOOK_NAME = 'slsqp-vs-lbfgs'
OUTPUT_DIR = get_output_dir(METHOD, NOTEBOOK_NAME, base='../output')
reset_figure_counter()
summary = log_run_header(METHOD, NOTEBOOK_NAME, OUTPUT_DIR)

  Benchmark  : slsqp-vs-lbfgs
  Method     : comparison
  Timestamp  : 2026-04-17T00:37:39
  Output dir : ..\output\comparison\slsqp-vs-lbfgs


## Helpers

In [3]:
import concurrent.futures

def init_stats_2d(deformation):
    H, W = deformation.shape[-2:]
    phi = np.stack([deformation[1, 0], deformation[2, 0]])
    j = jacobian_det2D(phi)[0]
    return int((j <= 0).sum()), float(j.min()), H * W, (H, W)

def final_stats_2d(phi_out, deformation):
    j = jacobian_det2D(phi_out)[0]
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    return int((j <= 0).sum()), float(j.min()), float(np.linalg.norm(phi_out - phi_init))

def init_stats_3d(dvf):
    j = jacobian_det3D(dvf)
    D, H, W = dvf.shape[1:]
    return int((j <= 0).sum()), float(j.min()), D * H * W, (D, H, W)

def final_stats_3d(phi, dvf):
    j = jacobian_det3D(phi)
    return int((j <= 0).sum()), float(j.min()), float(np.linalg.norm(phi - dvf))

def time_call(fn, *args, timeout=None, **kwargs):
    """Time fn(*args). If timeout (seconds) is set and fn exceeds it, raise TimeoutError.

    Python threads can't be forcibly killed, so on timeout the worker thread is
    abandoned and continues running until it hits a GIL-yielding point. That
    can contaminate the next test's timing, but it's the least-bad option on
    Windows + CUDA (subprocess teardown of torch state is painful).
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    if timeout is None:
        out = fn(*args, **kwargs)
    else:
        ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        future = ex.submit(fn, *args, **kwargs)
        try:
            out = future.result(timeout=timeout)
        except concurrent.futures.TimeoutError:
            raise TimeoutError(f'exceeded {timeout:.0f}s')
        finally:
            ex.shutdown(wait=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return out, time.time() - t0

In [4]:
def fullgrid_slsqp_2d(deformation, threshold=None, max_minimize_iter=None, verbose=0):
    """Single-shot SLSQP over the entire 2D grid (2*H*W variables, H*W constraints)."""
    threshold = threshold or DEFAULT_PARAMS['threshold']
    max_minimize_iter = max_minimize_iter or DEFAULT_PARAMS['max_minimize_iter']
    H, W = deformation.shape[-2:]
    pixels = H * W
    phi = np.stack([deformation[-2, 0], deformation[-1, 0]])  # (dy, dx)
    phi_flat = np.concatenate([phi[1].flatten(), phi[0].flatten()])  # [dx | dy]
    phi_init_flat = phi_flat.copy()

    def jac_con(z):
        dx = z[:pixels].reshape(H, W)
        dy = z[pixels:].reshape(H, W)
        return _numpy_jdet_2d(dy, dx).flatten()

    result = minimize(
        lambda z: objective_euc(z, phi_init_flat),
        phi_flat,
        jac=True,
        constraints=[NonlinearConstraint(jac_con, threshold, np.inf)],
        options={'maxiter': max_minimize_iter, 'disp': verbose >= 1},
        method='SLSQP',
    )
    out = np.zeros((2, H, W))
    out[1] = result.x[:pixels].reshape(H, W)
    out[0] = result.x[pixels:].reshape(H, W)
    return out

def fullgrid_slsqp_3d(dvf, threshold=None, max_minimize_iter=None, verbose=0):
    """Single-shot SLSQP over the entire 3D grid. Only viable on tiny volumes.

    Packing matches the 2D helper: phi_flat = [dx.flat | dy.flat | dz.flat]
    so we can reuse `objective_euc` unchanged.
    """
    threshold = threshold or DEFAULT_PARAMS['threshold']
    max_minimize_iter = max_minimize_iter or DEFAULT_PARAMS['max_minimize_iter']
    _, D, H, W = dvf.shape
    N = D * H * W
    dz, dy, dx = dvf[0], dvf[1], dvf[2]
    phi_flat = np.concatenate([dx.flatten(), dy.flatten(), dz.flatten()])
    phi_init_flat = phi_flat.copy()

    def jac_con(z):
        dx_ = z[:N].reshape(D, H, W)
        dy_ = z[N:2*N].reshape(D, H, W)
        dz_ = z[2*N:].reshape(D, H, W)
        return _numpy_jdet_3d(dz_, dy_, dx_).flatten()

    result = minimize(
        lambda z: objective_euc(z, phi_init_flat),
        phi_flat,
        jac=True,
        constraints=[NonlinearConstraint(jac_con, threshold, np.inf)],
        options={'maxiter': max_minimize_iter, 'disp': verbose >= 1},
        method='SLSQP',
    )
    out = np.zeros((3, D, H, W))
    out[2] = result.x[:N].reshape(D, H, W)
    out[1] = result.x[N:2*N].reshape(D, H, W)
    out[0] = result.x[2*N:].reshape(D, H, W)
    return out

## Configuration

Skip thresholds are tuned so the whole notebook finishes in a few minutes.
Raise them if you're willing to wait.

In [ ]:
# 2D: synthetic random DVFs (pure-noise baseline) + structured cases from
# test_cases (localized-fold synthetics via make_deformation) + real slices
# from load_slice. Real slices need absolute paths to the mpoints/fpoints
# .npy files since we're running from benchmarks/solvers/.
GRID_SIZES_2D = [10, 15, 20, 30, 40]
BASE_SHAPE_2D = (3, 1, 5, 5)
MAX_MAG_2D = 1.0
# Structured synthetic folds built by make_deformation (Laplacian from correspondences):
STRUCTURED_CASES_2D = [
    '01a_10x10_crossing',      # small crossing fold
    '01b_10x10_opposite',      # small opposite-motion fold
    '03b_10x10_crossing',      # larger-magnitude crossing
    '03c_20x20_opposite',      # medium opposite fold
    '01c_20x40_edges',         # edge-localized fold
]
# Real registration slices built by load_slice(slice_idx, scale_factor):
REAL_SLICES_2D = [
    # (label, slice_idx, scale_factor)
    ('02a_64x91',   90, 0.2),
    ('02c_64x91',  350, 0.2),
    ('02a_320x456', 90, 1.0),
]
MPOINTS_PATH = os.path.abspath('../../data/corrected_correspondences_count_touching/mpoints.npy')
FPOINTS_PATH = os.path.abspath('../../data/corrected_correspondences_count_touching/fpoints.npy')

# Full-grid SLSQP: 2*H*W vars + H*W dense constraints → KKT system blows up past 30x30.
SLSQP_FULL_MAX_PIXELS_2D = 900
# Windowed SLSQP: no pixel cap. Verified on 02a_320x456 (145,920 pix, 120 neg,
# min=-4.53) — completes in 0.73s because the bounding-window SLSQP sub-problems
# stay small (first QP was 9x9 = 162 vars). 2D can still stall on edge-pinned
# folds (e.g. 01c_20x40_edges takes ~25s) but does not hang.

# Per-test timeout (seconds). Applied uniformly to every method; if a solver
# exceeds this, the test is abandoned and we move to the next method. 5 min
# is enough for every case that will succeed — failures are hangs (SLSQP
# Fortran77 active-set cycling) and extending the timeout doesn't recover
# them, it just makes you wait longer for the same dead end.
TIMEOUT_SECONDS = 1800

# 3D: synthetic random DVFs + real downsampled volume. Random DVFs are
# capped at 8^3 — beyond that they stop being a meaningful baseline (pure
# i.i.d. noise scales badly and doesn't match real fold topology). Full-grid
# SLSQP skipped entirely for 3D. Windowed SLSQP runs on every 3D case —
# expect it to hang on real_6+ because 3D bounding windows grow one extra
# dim with +2*pad, pushing the first QP past SLSQP's cycling cliff
# (real_6 first QP: 1,755 vars, 585 cons — vs real_8: 840 vars, 280 cons).
GRID_SIZES_3D = [(8, 8, 8)]
BASE_SHAPE_3D = (3, 3, 3, 3)
MAX_MAG_3D = 4.0
SEED = 42
DOWNSAMPLE_FACTORS = [1/8, 1/6, 1/4, 1/3, 1/2]
FULL_VOLUME_PATH = os.path.abspath('../../data/corrected_correspondences_count_touching/registered_output/deformation3d.npy')

# GPU VRAM cap: (3 * D * H * W) active DOFs. Prevents OOM on an 8 GB card.
GPU_FULL_MAX_DOFS = 40_000_000

In [6]:
# Build 2D test fields: synthetic random DVFs, then structured folds,
# then real registration slices.
fields_2d = {}
base_2d = generate_random_dvf(BASE_SHAPE_2D, MAX_MAG_2D, SEED)
for s in GRID_SIZES_2D:
    dvf = scale_dvf(base_2d, (s, s))
    if init_stats_2d(dvf)[0] == 0:
        continue
    fields_2d[f'rand_{s}x{s}'] = dvf
for key in STRUCTURED_CASES_2D:
    deformation, *_ = make_deformation(key)
    if init_stats_2d(deformation)[0] == 0:
        print(f'  skip {key}: no neg-Jdet')
        continue
    fields_2d[key] = deformation
if os.path.exists(MPOINTS_PATH) and os.path.exists(FPOINTS_PATH):
    for (label, slice_idx, sf) in REAL_SLICES_2D:
        deformation, *_ = load_slice(
            slice_idx, sf,
            mpoints_path=MPOINTS_PATH, fpoints_path=FPOINTS_PATH)
        if init_stats_2d(deformation)[0] == 0:
            print(f'  skip {label}: no neg-Jdet')
            continue
        fields_2d[label] = deformation
else:
    print(f'[!] real slice points not found at {MPOINTS_PATH} — skipping 02_* cases')

# Build 3D test fields: synthetic random DVFs, then real downsampled volume.
fields_3d = {}
base_3d = generate_random_dvf_3d(BASE_SHAPE_3D, MAX_MAG_3D, SEED)
for shape in GRID_SIZES_3D:
    dvf = scale_dvf_3d(base_3d, shape)
    if init_stats_3d(dvf)[0] == 0:
        continue
    fields_3d[f'rand_{shape[0]}x{shape[1]}x{shape[2]}'] = dvf

if os.path.exists(FULL_VOLUME_PATH):
    full = np.load(FULL_VOLUME_PATH)
    _, Df, Hf, Wf = full.shape
    print(f'real volume loaded: shape={full.shape}')
    for factor in DOWNSAMPLE_FACTORS:
        new_shape = (max(1, int(round(Df * factor))),
                     max(1, int(round(Hf * factor))),
                     max(1, int(round(Wf * factor))))
        label = f'real_{int(round(1/factor))}'  # real_8, real_6, real_4, real_3, real_2
        fields_3d[label] = scale_dvf_3d(full, new_shape)
    del full
else:
    print(f'[!] real volume not found at {FULL_VOLUME_PATH} — using synthetic only')

print('\n2D cases:')
for k, f in fields_2d.items():
    n0, m0, nv, _ = init_stats_2d(f)
    print(f'  {k:<22s}  pix={nv:>7d}  neg0={n0:>5d}  min0={m0:+.4f}')
print('\n3D cases:')
for k, f in fields_3d.items():
    n0, m0, nv, _ = init_stats_3d(f)
    print(f'  {k:<14s}  shape={f.shape[1:]!s:<22s}  vox={nv:>8d}  neg0={n0:>5d}  min0={m0:+.4f}')

Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
Building data for Laplacian Sparse Matrix A (optimized)
Creating Laplacian Sparse Matrix A
real volume loaded: shape=(3, 528, 320, 456)

2D cases:
  rand_10x10              pix=    100  neg0=    5  min0=-0.8081
  rand_15x15              pix=    225  neg0=   16  min0=-0.7344
  rand_20x20              pix=    400  neg0=   27  min0=-0.6997
  rand_30x30            

## 2D benchmark

In [7]:
METHODS_2D = [
    ('slsqp_win',      'SLSQP win',         lambda d: iterative_serial(d.copy(), verbose=0)),
    ('slsqp_full',     'SLSQP full',        lambda d: fullgrid_slsqp_2d(d, verbose=0)),
    ('lbfgs_cpu_win',  'L-BFGS CPU win',    lambda d: iterative_2d_barrier(d, verbose=0, windowed=True)),
    ('lbfgs_cpu_full', 'L-BFGS CPU full',   lambda d: iterative_2d_barrier(d, verbose=0, windowed=False)),
    ('lbfgs_gpu_win',  'L-BFGS GPU win',    lambda d: iterative_2d_barrier_torch(d, verbose=0, windowed=True)),
    ('lbfgs_gpu_full', 'L-BFGS GPU full',   lambda d: iterative_2d_barrier_torch(d, verbose=0, windowed=False)),
]

rows_2d = {}
for key, dvf in fields_2d.items():
    init_neg, init_min, n_pix, shape = init_stats_2d(dvf)
    print(f'\n=== 2D {key}  shape={shape}  pix={n_pix}  neg0={init_neg}  min0={init_min:+.4f} ===')
    rows_2d[key] = {'shape': shape, 'n_vox': n_pix, 'init_neg': init_neg, 'init_min': init_min}
    for mkey, label, fn in METHODS_2D:
        skip_reason = None
        if mkey == 'slsqp_full' and n_pix > SLSQP_FULL_MAX_PIXELS_2D:
            skip_reason = f'pix>{SLSQP_FULL_MAX_PIXELS_2D}'
        if skip_reason:
            print(f'  {label:<18s}  [skipped: {skip_reason}]')
            rows_2d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}
            continue
        try:
            phi, t = time_call(fn, dvf, timeout=TIMEOUT_SECONDS)
            fn_, fm_, l2_ = final_stats_2d(phi, dvf)
            rows_2d[key][mkey] = {'t': t, 'neg': fn_, 'min': fm_, 'l2': l2_}
            print(f'  {label:<18s}  t={t:>8.2f}s  neg={fn_:>4d}  min={fm_:+.5f}  L2={l2_:.4f}')
        except TimeoutError as e:
            print(f'  {label:<18s}  [TIMEOUT {e}]')
            rows_2d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}
        except Exception as e:
            print(f'  {label:<18s}  [ERROR] {type(e).__name__}: {e}')
            rows_2d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}


=== 2D rand_10x10  shape=(10, 10)  pix=100  neg0=5  min0=-0.8081 ===
  SLSQP win           t=   25.12s  neg=   1  min=-0.07976  L2=0.5980
  SLSQP full          t=    0.12s  neg=   0  min=+0.01000  L2=0.6137
  L-BFGS CPU win      t=    0.13s  neg=   0  min=+0.01100  L2=0.6156
  L-BFGS CPU full     t=    0.06s  neg=   0  min=+0.01100  L2=0.6156
  L-BFGS GPU win      t=    3.15s  neg=   0  min=+0.01058  L2=0.6154
  L-BFGS GPU full     t=    1.01s  neg=   0  min=+0.01279  L2=0.6754

=== 2D rand_15x15  shape=(15, 15)  pix=225  neg0=16  min0=-0.7344 ===
  SLSQP win           t=    0.01s  neg=   0  min=+0.01000  L2=1.3419
  SLSQP full          t=    0.45s  neg=   0  min=+0.01000  L2=1.3419
  L-BFGS CPU win      t=    0.25s  neg=   0  min=+0.01100  L2=1.3480
  L-BFGS CPU full     t=    0.17s  neg=   0  min=+0.01100  L2=1.3473
  L-BFGS GPU win      t=    2.52s  neg=   0  min=+0.01100  L2=1.3481
  L-BFGS GPU full     t=    2.03s  neg=   0  min=+0.01000  L2=1.4261

=== 2D rand_20x20  shape=(20, 

## 3D benchmark

In [ ]:
# No full-grid SLSQP in 3D — 3*N^2 constraint-Jacobian cost makes it
# unusable even at 8^3, and it's not a method anyone would reach for.
#
# SLSQP windowed starts at max_window_voxels=400 (total voxel count of
# every sub-volume, aspect-preserving shrink when the bounding box
# exceeds budget). SLSQP's cost scales O((3V + V)^2) with V = window
# voxels. A per-axis cap like (7,7,7) is the same 343-voxel budget but
# rejects safe elongated shapes like (3,3,39) = 351 voxels.
#
# Without any cap the first QP at real_6 would be 1,755 vars x 585 cons
# (scripts/_probe_slsqp_real6.py) and SLSQP's Fortran77 active-set QP
# hangs in a cycling regime. With a static cap the solver always
# returns but livelocks on dense folds — the worst voxel is re-picked
# each outer iter and neg-count oscillates around a plateau.
#
# Escalation (max_window_voxels_ceiling): after voxel_cap_stall_threshold
# iters without a new best neg-count, the cap is multiplied by
# voxel_cap_growth up to the ceiling. This gives SLSQP a larger window
# to escape livelock. Trade-off: bigger windows move closer to SLSQP's
# hang regime (empirically ~500-600 voxels on our real_6 case). We keep
# the ceiling at 500 — one modest 400->500 bump — and rely on
# TIMEOUT_SECONDS as the ultimate safety net. If you need genuine
# convergence on dense folds, switch to L-BFGS barrier.
#
# Small-grid exception: when the whole grid is <= the cap, carving it
# into windows creates livelock where none existed — SLSQP can just
# solve the full 8^3 = 512 vox problem in one shot. So skip the cap
# whenever (D*H*W) <= SLSQP_3D_SMALL_GRID_VOXELS.
SLSQP_3D_MAX_WINDOW_VOXELS = 400
SLSQP_3D_MAX_WINDOW_VOXELS_CEILING = 500
SLSQP_3D_SMALL_GRID_VOXELS = 1000  # below this, let SLSQP solve full-grid
SLSQP_3D_MAX_ITERATIONS = 500  # benchmark wall-time safety cap

def _slsqp_win_3d(d):
    n_vox = d.shape[1] * d.shape[2] * d.shape[3]
    if n_vox <= SLSQP_3D_SMALL_GRID_VOXELS:
        # Full-grid SLSQP sub-problem is tractable; avoid windowed livelock.
        return iterative_3d(d.copy(), verbose=0,
                            max_iterations=SLSQP_3D_MAX_ITERATIONS)
    return iterative_3d(
        d.copy(), verbose=0,
        max_window_voxels=SLSQP_3D_MAX_WINDOW_VOXELS,
        max_window_voxels_ceiling=SLSQP_3D_MAX_WINDOW_VOXELS_CEILING,
        max_iterations=SLSQP_3D_MAX_ITERATIONS,
    )

METHODS_3D = [
    ('slsqp_win',      'SLSQP win',         _slsqp_win_3d),
    ('lbfgs_cpu_win',  'L-BFGS CPU win',    lambda d: iterative_3d_barrier(d, verbose=0, windowed=True)),
    ('lbfgs_cpu_full', 'L-BFGS CPU full',   lambda d: iterative_3d_barrier(d, verbose=0, windowed=False)),
    ('lbfgs_gpu_win',  'L-BFGS GPU win',    lambda d: iterative_3d_barrier_torch(d, verbose=0, windowed=True)),
    ('lbfgs_gpu_full', 'L-BFGS GPU full',   lambda d: iterative_3d_barrier_torch(d, verbose=0, windowed=False)),
]

rows_3d = {}
for key, dvf in fields_3d.items():
    init_neg, init_min, n_vox, shape = init_stats_3d(dvf)
    n_dofs = 3 * n_vox
    print(f'\n=== 3D {key}  shape={shape}  vox={n_vox}  neg0={init_neg}  min0={init_min:+.4f} ===')
    rows_3d[key] = {'shape': shape, 'n_vox': n_vox, 'init_neg': init_neg, 'init_min': init_min}
    for mkey, label, fn in METHODS_3D:
        skip_reason = None
        if mkey == 'lbfgs_gpu_full' and n_dofs > GPU_FULL_MAX_DOFS:
            skip_reason = f'DOFs>{GPU_FULL_MAX_DOFS}'
        if skip_reason:
            print(f'  {label:<18s}  [skipped: {skip_reason}]')
            rows_3d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}
            continue
        try:
            phi, t = time_call(fn, dvf, timeout=TIMEOUT_SECONDS)
            fn_, fm_, l2_ = final_stats_3d(phi, dvf)
            rows_3d[key][mkey] = {'t': t, 'neg': fn_, 'min': fm_, 'l2': l2_}
            print(f'  {label:<18s}  t={t:>8.2f}s  neg={fn_:>4d}  min={fm_:+.5f}  L2={l2_:.4f}')
        except TimeoutError as e:
            print(f'  {label:<18s}  [TIMEOUT {e}]')
            rows_3d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}
        except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
            print(f'  {label:<18s}  [OOM/RT] {e}')
            rows_3d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f'  {label:<18s}  [ERROR] {type(e).__name__}: {e}')
            rows_3d[key][mkey] = {'t': np.nan, 'neg': -1, 'min': np.nan, 'l2': np.nan}


=== 3D rand_8x8x8  shape=(8, 8, 8)  vox=512  neg0=244  min0=-43.2706 ===
  SLSQP win           t=  123.49s  neg=   0  min=+0.01000  L2=10.3818
  L-BFGS CPU win      t=    1.08s  neg=   0  min=+0.01100  L2=10.3928
  L-BFGS CPU full     t=    0.93s  neg=   0  min=+0.01100  L2=10.3945
  L-BFGS GPU win      t=   10.54s  neg=   0  min=+0.01100  L2=10.3918
  L-BFGS GPU full     t=    9.68s  neg=   0  min=+0.01100  L2=10.3921

=== 3D real_8  shape=(66, 40, 57)  vox=150480  neg0=32  min0=-1.8397 ===
  SLSQP win           t=    6.59s  neg=   0  min=+0.01000  L2=2.2842
  L-BFGS CPU win      t=    1.03s  neg=   0  min=+0.01100  L2=2.2927
  L-BFGS CPU full     t=   75.68s  neg=   0  min=+0.01100  L2=2.2883
  L-BFGS GPU win      t=    6.77s  neg=   0  min=+0.01003  L2=2.2883
  L-BFGS GPU full     t=    3.02s  neg=   0  min=+0.01100  L2=2.2884

=== 3D real_6  shape=(88, 53, 76)  vox=354464  neg0=83  min0=-4.2676 ===
  SLSQP win           [TIMEOUT exceeded 300s]
  L-BFGS CPU win      t=   52.90s  ne

## Summary tables

In [ ]:
def _fmt_t(v):
    return f'{v:>8.2f}' if isinstance(v, (int, float)) and np.isfinite(v) else f'{"-":>8s}'
def _fmt_l2(v):
    return f'{v:>8.4f}' if isinstance(v, (int, float)) and np.isfinite(v) else f'{"-":>8s}'
def _fmt_neg(v):
    return f'{v:>5d}' if isinstance(v, int) and v >= 0 else f'{"-":>5s}'

# Separate key/label lists per dim: 3D drops slsqp_full (not tested).
METHOD_KEYS_2D = [m[0] for m in METHODS_2D]
METHOD_LABELS_2D = [m[1] for m in METHODS_2D]
METHOD_KEYS_3D = [m[0] for m in METHODS_3D]
METHOD_LABELS_3D = [m[1] for m in METHODS_3D]

def print_table(rows, keys, labels, title):
    print(title)
    hdr = f'{"case":<22s} {"neg0":>5s}'
    for lab in labels:
        hdr += f'  {lab+" t":>12s} {lab+" neg":>7s} {lab+" L2":>9s}'
    print(hdr)
    for k, r in rows.items():
        line = f'{k:<22s} {r["init_neg"]:>5d}'
        for mkey in keys:
            m = r[mkey]
            line += f'  {_fmt_t(m["t"]):>12s} {_fmt_neg(m["neg"]):>7s} {_fmt_l2(m["l2"]):>9s}'
        print(line)

print_table(rows_2d, METHOD_KEYS_2D, METHOD_LABELS_2D, '\n--- 2D summary ---')
print_table(rows_3d, METHOD_KEYS_3D, METHOD_LABELS_3D, '\n--- 3D summary ---')

## Wall-time comparison

In [ ]:
COLORS = {
    'slsqp_win':      'tab:blue',
    'slsqp_full':     'tab:red',
    'lbfgs_cpu_win':  'tab:cyan',
    'lbfgs_cpu_full': 'tab:purple',
    'lbfgs_gpu_win':  'tab:orange',
    'lbfgs_gpu_full': 'tab:green',
}

def plot_walltime(rows, keys, labels, title, ax):
    cases = list(rows.keys())
    x = np.arange(len(cases))
    n_methods = len(keys)
    w = 0.8 / n_methods
    for i, (mkey, lab) in enumerate(zip(keys, labels)):
        t = [rows[c][mkey]['t'] for c in cases]
        t = [v if (isinstance(v, float) and np.isfinite(v)) else np.nan for v in t]
        ax.bar(x + (i - (n_methods - 1) / 2) * w, t, w,
               label=lab, color=COLORS[mkey])
    ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(cases, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Time (s, log)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=8, ncol=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
plot_walltime(rows_2d, METHOD_KEYS_2D, METHOD_LABELS_2D, '2D wall time', axes[0])
plot_walltime(rows_3d, METHOD_KEYS_3D, METHOD_LABELS_3D, '3D wall time', axes[1])
plt.tight_layout()
show_and_save(OUTPUT_DIR, name='walltime')

## L2 distortion comparison

Lower = the corrector changed the input field less. Full-grid methods generally produce lower L2 because the correction is distributed globally rather than concentrated in frozen-edge windows.

In [ ]:
def plot_l2(rows, keys, labels, title, ax):
    cases = list(rows.keys())
    x = np.arange(len(cases))
    n_methods = len(keys)
    w = 0.8 / n_methods
    for i, (mkey, lab) in enumerate(zip(keys, labels)):
        l2 = [rows[c][mkey]['l2'] for c in cases]
        l2 = [v if (isinstance(v, float) and np.isfinite(v)) else np.nan for v in l2]
        ax.bar(x + (i - (n_methods - 1) / 2) * w, l2, w,
               label=lab, color=COLORS[mkey])
    ax.set_xticks(x)
    ax.set_xticklabels(cases, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('L2 distortion ||phi - phi_init||')
    ax.set_title(title)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=8, ncol=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
plot_l2(rows_2d, METHOD_KEYS_2D, METHOD_LABELS_2D, '2D L2 distortion', axes[0])
plot_l2(rows_3d, METHOD_KEYS_3D, METHOD_LABELS_3D, '3D L2 distortion', axes[1])
plt.tight_layout()
show_and_save(OUTPUT_DIR, name='l2_distortion')

## Scaling: wall time vs problem size

Log-log plot reveals the asymptotic cost of each method. Full-grid SLSQP
should have the steepest slope; GPU windowed L-BFGS should flatten.

In [ ]:
def plot_scaling(rows, keys, labels, title, ax):
    vox = [rows[c]['n_vox'] for c in rows]
    for mkey, lab in zip(keys, labels):
        t = [rows[c][mkey]['t'] for c in rows]
        xs, ys = [], []
        for v, tv in zip(vox, t):
            if isinstance(tv, float) and np.isfinite(tv):
                xs.append(v); ys.append(tv)
        if xs:
            ax.loglog(xs, ys, 'o-', label=lab, color=COLORS[mkey])
    ax.set_xlabel('Voxels (total)')
    ax.set_ylabel('Time (s)')
    ax.set_title(title)
    ax.grid(True, which='both', alpha=0.3)
    ax.legend(fontsize=8, ncol=2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
plot_scaling(rows_2d, METHOD_KEYS_2D, METHOD_LABELS_2D, '2D scaling', axes[0])
plot_scaling(rows_3d, METHOD_KEYS_3D, METHOD_LABELS_3D, '3D scaling', axes[1])
plt.tight_layout()
show_and_save(OUTPUT_DIR, name='scaling')

## Speedup heatmap (time ratio, SLSQP windowed = 1x baseline)

Each cell is `t(SLSQP win) / t(method)`. Values > 1 mean faster than SLSQP windowed; NaN cells were skipped.

In [ ]:
def speedup_matrix(rows, keys):
    cases = list(rows.keys())
    M = np.full((len(cases), len(keys)), np.nan)
    for i, c in enumerate(cases):
        t_base = rows[c]['slsqp_win']['t']
        if not (isinstance(t_base, float) and np.isfinite(t_base)):
            continue
        for j, mkey in enumerate(keys):
            t = rows[c][mkey]['t']
            if isinstance(t, float) and np.isfinite(t) and t > 0:
                M[i, j] = t_base / t
    return cases, M

def plot_speedup(rows, keys, labels, title, ax):
    cases, M = speedup_matrix(rows, keys)
    logM = np.log10(M)
    im = ax.imshow(logM, cmap='RdBu', vmin=-2, vmax=2, aspect='auto')
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticks(range(len(cases)))
    ax.set_yticklabels(cases, fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            txt = '-' if not np.isfinite(v) else (f'{v:.1f}x' if v >= 1 else f'{v:.2f}x')
            ax.text(j, i, txt, ha='center', va='center', fontsize=8,
                    color='black' if (np.isfinite(v) and abs(np.log10(v)) < 1) else 'white')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label='log10 speedup')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_speedup(rows_2d, METHOD_KEYS_2D, METHOD_LABELS_2D, '2D speedup vs SLSQP windowed', axes[0])
plot_speedup(rows_3d, METHOD_KEYS_3D, METHOD_LABELS_3D, '3D speedup vs SLSQP windowed', axes[1])
plt.tight_layout()
show_and_save(OUTPUT_DIR, name='speedup_heatmap')

## Save results

In [ ]:
def build_columns(keys):
    cols = ['case', 'shape', 'n_vox', 'init_neg', 'init_min']
    for mkey in keys:
        cols += [f't_{mkey}', f'neg_{mkey}', f'min_{mkey}', f'l2_{mkey}']
    return cols

def row_to_list(key, r, keys):
    sh = 'x'.join(str(s) for s in r['shape'])
    out = [key, sh, r['n_vox'], r['init_neg'], round(r['init_min'], 6)]
    for mkey in keys:
        m = r[mkey]
        def _r(v, d=4):
            return round(v, d) if isinstance(v, (int, float)) and np.isfinite(v) else ''
        out += [_r(m['t'], 4), m['neg'] if m['neg'] >= 0 else '',
                _r(m['min'], 6), _r(m['l2'], 6)]
    return out

if rows_2d:
    save_results_csv([row_to_list(k, r, METHOD_KEYS_2D) for k, r in rows_2d.items()],
                     build_columns(METHOD_KEYS_2D), OUTPUT_DIR, name='results_2d')
if rows_3d:
    save_results_csv([row_to_list(k, r, METHOD_KEYS_3D) for k, r in rows_3d.items()],
                     build_columns(METHOD_KEYS_3D), OUTPUT_DIR, name='results_3d')

combined = {}
def _add(tag, rows, keys, labels):
    for k, r in rows.items():
        for mkey, lab in zip(keys, labels):
            m = r[mkey]
            combined[f'{tag} {k} [{lab}]'] = {
                'n_neg_init': r['init_neg'],
                'n_neg_final': m['neg'] if m['neg'] >= 0 else 0,
                'min_jdet_init': r['init_min'],
                'min_jdet': m['min'] if isinstance(m['min'], float) and np.isfinite(m['min']) else 0.0,
                'l2_err': m['l2'] if isinstance(m['l2'], float) and np.isfinite(m['l2']) else 0.0,
                'time': m['t'] if isinstance(m['t'], float) and np.isfinite(m['t']) else 0.0,
            }
_add('2D', rows_2d, METHOD_KEYS_2D, METHOD_LABELS_2D)
_add('3D', rows_3d, METHOD_KEYS_3D, METHOD_LABELS_3D)
summary = log_run_footer(summary, combined)
save_summary_json(summary, OUTPUT_DIR)